# Compound AI Systems Exercise — Assessment 03

**Role context:** Researcher interview, live notebook prototyping  
**Time limit: 45 minutes. No AI coding assistance.**


## Problem Statement

Procurement receives **messy vendor emails** that mix order changes, corrections, warranty lines, and unrelated accounting noise. Buyers need a **structured purchase-order draft** plus a list of uncertainties to confirm before ERP entry.

Build a compound system that turns one or more related emails into a validated PO draft.


## Functional Requirements

**Input**
- `emails: list[dict]` with keys `email_id`, `from`, `subject`, `body`
- Emails may be multi-turn; later messages can correct earlier ones.

**Output**
```python
{
  "is_purchase_order_request": true/false,
  "po_draft": {
    "vendor": "...",
    "bill_to": "...",
    "ship_to": "...",
    "payment_terms": "...",
    "requested_delivery": "...",
    "line_items": [
      {
        "sku": "...",
        "description": "...",
        "quantity": 0,
        "unit_price": 0.0,
        "line_total": 0.0,
        "charge_type": "standard" | "warranty" | "rush" | "other"
      }
    ],
    "currency": "USD",
    "subtotal": 0.0
  },
  "corrections_applied": ["human-readable notes about superseded quantities/etc"],
  "needs_human_review": [
    {"field": "...", "reason": "...", "priority": "high"|"medium"|"low"}
  ],
  "ignored_emails": ["email ids that are not order-related"],
  "validation_notes": ["arithmetic or consistency checks you ran"]
}
```

If `is_purchase_order_request` is false, `po_draft` may be `null`.

**Constraints**
- Apply corrections chronologically when emails revise earlier quantities.
- Recompute `line_total` and `subtotal`; do not trust the email's arithmetic blindly.
- Warranty $0 lines should still appear if they request shipment.
- Flag missing SKUs, ambiguous dates, or conflicting ship-to as human review items.

**Edge cases**
- Non-order emails in the thread.
- Explicit "ignore my earlier number".
- Mix of paid and warranty lines.

**Desired behavior**
A buyer should paste `po_draft` into ERP after clearing `needs_human_review`.


## Available Tools & Starter Code


In [ ]:
import json
import os
import re
from typing import Any, Optional

# ---------------------------------------------------------------------------
# LLM configuration
# ---------------------------------------------------------------------------
# Set one of the following before running (or export in your shell):
#
#   export OPENAI_API_KEY="sk-..."
#   # optional: export OPENAI_MODEL="gpt-4o-mini"
#
#   export ANTHROPIC_API_KEY="sk-ant-..."
#   # optional: export ANTHROPIC_MODEL="claude-3-5-haiku-latest"
#
# If neither key is set, call_llm runs in MOCK mode so the notebook still
# executes for smoke-testing. Mock responses are placeholders — they are not
# a substitute for a real model during practice.
# ---------------------------------------------------------------------------

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "").strip()
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
ANTHROPIC_MODEL = os.environ.get("ANTHROPIC_MODEL", "claude-3-5-haiku-latest")

USE_MOCK = not (OPENAI_API_KEY or ANTHROPIC_API_KEY)
if USE_MOCK:
    print("⚠️  No API key found. call_llm is in MOCK mode.")
else:
    provider = "OpenAI" if OPENAI_API_KEY else "Anthropic"
    print(f"✓ Using {provider} API")


def _mock_response(prompt: str, system_prompt: str = "") -> str:
    """Lightweight deterministic mock for offline smoke tests."""
    combined = f"{system_prompt}\n{prompt}".lower()
    # Return minimal JSON-ish or text so downstream parsing can be exercised.
    if "json" in combined:
        return json.dumps({"mock": True, "note": "Replace with real API key for useful output", "items": []})
    return "[MOCK RESPONSE] Configure OPENAI_API_KEY or ANTHROPIC_API_KEY for real model output."


def call_llm(
    prompt: str,
    system_prompt: str = "You are a helpful assistant.",
    temperature: float = 0.2,
    max_tokens: int = 2000,
) -> str:
    """Call a chat LLM. Credentials come from environment variables only."""
    if USE_MOCK:
        return _mock_response(prompt, system_prompt)

    try:
        if OPENAI_API_KEY:
            from openai import OpenAI

            client = OpenAI(api_key=OPENAI_API_KEY)
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                temperature=temperature,
                max_tokens=max_tokens,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ],
            )
            return (resp.choices[0].message.content or "").strip()

        from anthropic import Anthropic

        client = Anthropic(api_key=ANTHROPIC_API_KEY)
        resp = client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=max_tokens,
            temperature=temperature,
            system=system_prompt,
            messages=[{"role": "user", "content": prompt}],
        )
        return "".join(
            block.text for block in resp.content if getattr(block, "type", None) == "text"
        ).strip()
    except Exception as e:
        return f"[API ERROR] {type(e).__name__}: {e}"


def extract_json(text: str) -> Any:
    """Best-effort parse of a JSON object/array from model output."""
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    fence = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if fence:
        try:
            return json.loads(fence.group(1).strip())
        except json.JSONDecodeError:
            pass
    for opener, closer in (("{", "}"), ("[", "]")):
        start, end = text.find(opener), text.rfind(closer)
        if start != -1 and end > start:
            try:
                return json.loads(text[start : end + 1])
            except json.JSONDecodeError:
                continue
    raise ValueError(f"Could not parse JSON from model output:\n{text[:500]}")


# Smoke test
print("call_llm ready.")
print(call_llm("Reply with exactly: OK", system_prompt="Be terse.")[:80])


In [ ]:
VENDOR_EMAILS = [
  {
    "email_id": "E-1001",
    "from": "sales@northwind-components.com",
    "subject": "Re: quote follow-up / your PO?",
    "body": "Hi team,\n\nThanks for the call. As discussed please ship:\n- 40x NW-44A mounting brackets @ $12.50 each (same as last quarter)\n- 15x NW-90 cable kits \u2014 wait, making that 18 if you can, @ $27 each\n- 2x rush replacement for NW-12 seals, warranty, $0\n\nDeliver to Warehouse B, 880 Industrial Pkwy, Dock 3 by Friday next week if possible.\nBill to Acme Robotics, Net 30, attention: AP.\n\nAlso ignore my earlier email that said 25 brackets \u2014 that was wrong.\n\nThanks,\nMina (Northwind)\n"
  },
  {
    "email_id": "E-1002",
    "from": "accounts@northwind-components.com",
    "subject": "Invoice reminder \u2014 unrelated",
    "body": "Friendly reminder invoice INV-8891 for $420 is due. Please remit. This is not a new order."
  }
]

print("Thread size:", len(VENDOR_EMAILS))


## Coding Section


In [ ]:
def build_po_draft(emails: list) -> dict:
    """Convert vendor email thread(s) into a structured PO draft.

    Args:
        emails: list of email dicts (email_id, from, subject, body).

    Returns:
        dict matching Functional Requirements.
    """
    # TODO: Implement your compound AI system here.
    raise NotImplementedError("Implement build_po_draft")


## Example Input / Output

For `VENDOR_EMAILS`, expect roughly:
- Brackets quantity **40** (not 25)
- Cable kits **18** @ $27
- Warranty seals qty **2** @ $0
- E-1002 listed under `ignored_emails`
- Subtotal ≈ 40*12.50 + 18*27 + 0

Exact wording of review notes may vary.


## Run Your Solution


In [ ]:
# result = build_po_draft(VENDOR_EMAILS)
# print(json.dumps(result, indent=2))


## Time Limit

**Time limit: 45 minutes. No AI coding assistance.**
